# 05 - ML Temperature Prediction

Phase 1 benchmark for the residential thermal digital twin.

Goal: predict indoor temperature 1 hour ahead for:
- `temp_salon_c`
- `temp_kids_c`

Using:
- indoor temperature/humidity
- outdoor north/south temperatures/humidity
- thermal lag features
- time-cycle features
- solar proxy: `temp_out_2_n_c - temp_out_1_b_c`

This notebook uses the hourly dataset exported from the Home Assistant.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.multioutput import MultiOutputRegressor


## 1. Load hourly dataset

In [2]:
DATA_PATH = Path("../data/processed/TempHumData_27_4_26.csv")

# If running this notebook from a different location, update DATA_PATH accordingly.
df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()

cutoff = "2026-02-22 17:00:00"
df = df[df.index >= cutoff]

print(df.shape)
display(df.head())
display(df.tail())

(1524, 8)


,temp_salon_c,temp_kids_c,hum_salon,hum_kids,temp_out_1_b_c,hum_out_1_b,temp_out_2_n_c,hum_out_2_n
time,,,,,,,,
2026-02-22 17:00:00,18.45,19.84,63.91,55.66,9.89,66.82,17.01,55.45
2026-02-22 18:00:00,19.33,20.04,63.81,56.66,9.23,70.83,11.63,63.32
2026-02-22 19:00:00,19.07,19.65,65.78,60.08,8.89,70.79,10.33,66.01
2026-02-22 20:00:00,19.44,19.84,66.32,61.13,8.64,71.44,9.66,68.00
2026-02-22 21:00:00,19.26,19.54,65.34,61.40,8.44,75.14,9.39,71.99


,temp_salon_c,temp_kids_c,hum_salon,hum_kids,temp_out_1_b_c,hum_out_1_b,temp_out_2_n_c,hum_out_2_n
time,,,,,,,,
2026-04-27 01:00:00,20.5,20.5,48.06,45.62,18.49,34.92,17.34,36.65
2026-04-27 02:00:00,20.5,20.5,47.60,46.10,17.97,36.76,16.76,38.86
2026-04-27 03:00:00,20.5,20.5,47.60,46.10,17.49,39.41,16.29,41.62
2026-04-27 04:00:00,20.5,20.5,47.60,47.02,17.03,42.39,15.92,44.46
2026-04-27 05:00:00,20.5,20.5,47.60,47.10,16.72,44.56,15.65,46.76


## 2. Quick data quality check

In [3]:
print("Date range:", df.index.min(), "→", df.index.max())
print("\nMissing values:")
display(df.isna().sum())

# Verify hourly spacing
time_diffs = df.index.to_series().diff().dropna()
display(time_diffs.value_counts().head())


Date range: 2026-02-22 17:00:00 → 2026-04-27 05:00:00

Missing values:


temp_salon_c      0
temp_kids_c       4
hum_salon         0
hum_kids          4
temp_out_1_b_c    0
hum_out_1_b       0
temp_out_2_n_c    0
hum_out_2_n       0
dtype: int64

time
0 days 01:00:00    1522
0 days 02:00:00       1
Name: count, dtype: int64

## 3. Clean and interpolate small gaps

In [4]:
# Hourly grid, in case any timestamps are missing
df = df.resample("1h").mean()

# Interpolate short gaps only. Long initial gaps in outdoor sensors remain NaN and will be dropped later.
df = df.interpolate(methods="time", limit=4)

# Optional: add mean outdoor conditions if both sides exist
df["temp_out_mean_c"] = df[["temp_out_1_b_c", "temp_out_2_n_c"]].mean(axis=1)
df["hum_out_mean"] = df[["hum_out_1_b", "hum_out_2_n"]].mean(axis=1)

# Solar / orientation proxy: south-facing outdoor sensor minus north-facing outdoor sensor.
# Confirm which outdoor sensor is north/south in your setup before interpreting physically.
df["solar_proxy_temp_delta"] = df["temp_out_2_n_c"] - df["temp_out_1_b_c"]

display(df.tail())

,temp_salon_c,temp_kids_c,hum_salon,hum_kids,temp_out_1_b_c,hum_out_1_b,temp_out_2_n_c,hum_out_2_n,temp_out_mean_c,hum_out_mean,solar_proxy_temp_delta
time,,,,,,,,,,,
2026-04-27 01:00:00,20.5,20.5,48.06,45.62,18.49,34.92,17.34,36.65,17.915,35.785,-1.15
2026-04-27 02:00:00,20.5,20.5,47.60,46.10,17.97,36.76,16.76,38.86,17.365,37.810,-1.21
2026-04-27 03:00:00,20.5,20.5,47.60,46.10,17.49,39.41,16.29,41.62,16.890,40.515,-1.20
2026-04-27 04:00:00,20.5,20.5,47.60,47.02,17.03,42.39,15.92,44.46,16.475,43.425,-1.11
2026-04-27 05:00:00,20.5,20.5,47.60,47.10,16.72,44.56,15.65,46.76,16.185,45.660,-1.07


## 4. Feature Engineering

In [7]:
def add_lag_features(data: pd.DataFrame, columns: list[str], lags: list[int]) -> pd.DataFrame:
    out = data.copy()
    for col in columns:
        for lag in lags:
            out[f"{col}_lag_{lag}h"] = out[col].shift(lag)
    return out

def add_time_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    hour = out.index.hour
    dayofyear = out.index.dayofyear

    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    out["doy_sin"] = np.sin(2 * np.pi * dayofyear / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * dayofyear / 365.25)

    out["is_weekend"] = (out.index.dayofweek >= 5).astype(int)

    return out

def add_rolling_features(data: pd.DataFrame, columns: list[str], windows: list[int]) -> pd.DataFrame:
    out = data.copy()
    for col in columns:
        for window in windows:
            out[f"{col}_roll_mean_{window}h"] = out[col].rolling(window).mean()
            out[f"{col}_roll_std_{window}h"] = out[col].rolling(window).std()
    return out

In [8]:
targets = ["temp_salon_c", "temp_kids_c"]
base_series_for_lags = [
    "temp_salon_c",
    "temp_kids_c",
    "temp_out_1_b_c",
    "temp_out_2_n_c",
    "temp_out_mean_c",
    "solar_proxy_temp_delta",
]

horizon_hours = 1
lags = [1, 2, 3, 6, 12, 24]
rolling_windows = [3, 6, 12, 24]

feat = add_time_features(df)
feat = add_lag_features(feat, base_series_for_lags, lags)
feat = add_rolling_features(feat, ["temp_salon_c", "temp_kids_c", "temp_out_mean_c"], rolling_windows)

for target in targets:
    feat[f"target_{target}_plus_{horizon_hours}h"] = feat[target].shift(-horizon_hours)

feat = feat.dropna()

print(feat.shape)
display(feat.head())


(1500, 78)


,temp_salon_c,temp_kids_c,hum_salon,hum_kids,temp_out_1_b_c,hum_out_1_b,temp_out_2_n_c,hum_out_2_n,temp_out_mean_c,hum_out_mean,...,temp_out_mean_c_roll_mean_3h,temp_out_mean_c_roll_std_3h,temp_out_mean_c_roll_mean_6h,temp_out_mean_c_roll_std_6h,temp_out_mean_c_roll_mean_12h,temp_out_mean_c_roll_std_12h,temp_out_mean_c_roll_mean_24h,temp_out_mean_c_roll_std_24h,target_temp_salon_c_plus_1h,target_temp_kids_c_plus_1h
time,,,,,,,,,,,,,,,,,,,,,
2026-02-23 17:00:00,17.88,19.75,61.50,53.01,13.00,55.39,20.00,38.42,16.500,46.905,...,16.981667,0.569349,15.933333,1.350451,13.002083,3.385965,10.812083,3.285524,19.05,20.18
2026-02-23 18:00:00,19.05,20.18,60.53,52.50,12.04,59.13,15.38,47.85,13.710,53.490,...,15.940000,2.009403,15.908333,1.397092,13.435417,3.077504,10.948750,3.336758,18.88,19.46
2026-02-23 19:00:00,18.88,19.46,61.02,54.42,11.24,62.39,13.52,53.81,12.380,58.100,...,14.196667,2.102673,15.470000,2.012394,13.763750,2.680696,11.064167,3.336345,18.67,19.23
2026-02-23 20:00:00,18.67,19.23,62.52,59.87,10.67,64.82,12.50,57.58,11.585,61.200,...,12.558333,1.073666,14.770000,2.541757,13.953750,2.402286,11.165625,3.312543,18.50,19.10
2026-02-23 21:00:00,18.50,19.10,65.08,62.20,10.22,66.57,11.59,60.82,10.905,63.695,...,11.623333,0.738247,13.781667,2.724549,14.012083,2.309696,11.248542,3.278489,18.31,18.90


## 5. Train/test split — chronological, not random

In [ ]:
target_cols = [f"target_{target}_plus_{horizon_hours}h" for target in targets]
feature_cols = [c for c in feat.columns if c not in target_cols]

X = feat[feature_cols]
y = feat[target_cols]

split_idx = int(len(feat) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Train:", X_train.index.min(), "→", X_train.index.max(), X_train.shape)
print("Test: ", X_test.index.min(), "→", X_test.index.max(), X_test.shape)


## 6. Baseline: persistence model

Persistence means: the best guess for `T(t+1h)` is simply `T(t)`.

In [ ]:
baseline_pred = pd.DataFrame(
    {
        f"target_temp_salon_c_plus_{horizon_hours}h": X_test["temp_salon_c"],
        f"target_temp_kids_c_plus_{horizon_hours}h": X_test["temp_kids_c"],
    },
    index=X_test.index,
)

for col in target_cols:
    mae = mean_absolute_error(y_test[col], baseline_pred[col])
    rmse = mean_squared_error(y_test[col], baseline_pred[col], squared=False)
    print(f"Persistence {col}: MAE={mae:.3f} °C | RMSE={rmse:.3f} °C")


## 7. Random Forest model

In [ ]:
rf = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
rf_pred = pd.DataFrame(rf.predict(X_test), index=X_test.index, columns=target_cols)

for col in target_cols:
    mae = mean_absolute_error(y_test[col], rf_pred[col])
    rmse = mean_squared_error(y_test[col], rf_pred[col], squared=False)
    print(f"Random Forest {col}: MAE={mae:.3f} °C | RMSE={rmse:.3f} °C")


## 8. Gradient Boosting model

In [ ]:
# HistGradientBoostingRegressor is available in scikit-learn and is a good non-extra-dependency benchmark.
gbr = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.04,
        l2_regularization=0.01,
        random_state=42,
    )
)

gbr.fit(X_train, y_train)
gbr_pred = pd.DataFrame(gbr.predict(X_test), index=X_test.index, columns=target_cols)

for col in target_cols:
    mae = mean_absolute_error(y_test[col], gbr_pred[col])
    rmse = mean_squared_error(y_test[col], gbr_pred[col], squared=False)
    print(f"HistGradientBoosting {col}: MAE={mae:.3f} °C | RMSE={rmse:.3f} °C")


## 9. Plot predictions

In [ ]:
def plot_prediction(y_true, y_pred, target_col, title, n=240):
    plt.figure(figsize=(14, 5))
    plt.plot(y_true[target_col].iloc[:n].index, y_true[target_col].iloc[:n], label="Actual")
    plt.plot(y_pred[target_col].iloc[:n].index, y_pred[target_col].iloc[:n], label="Predicted")
    plt.title(title)
    plt.ylabel("Temperature [°C]")
    plt.xlabel("Time")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_prediction(y_test, rf_pred, target_cols[0], "Random Forest — Salon temperature +1h")
plot_prediction(y_test, rf_pred, target_cols[1], "Random Forest — Kids room temperature +1h")


## 10. Feature importance

In [ ]:
# Built-in feature importance for Random Forest.
importance = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": rf.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(importance.head(25))


## 11. Save processed feature dataset

In [ ]:
OUT_PATH = Path("../data/processed/ml_hourly_features.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

feat.to_csv(OUT_PATH)
print("Saved:", OUT_PATH)
